# Carga de Titanic-Sencillo.csv
En este cuaderno se carga el archivo `Titanic-Sencillo.csv` y se muestra la información del DataFrame.

```
CREATE TABLE IF NOT EXISTS `pasajeros` (
  `passengerid` int unsigned NOT NULL,
  `age` tinyint unsigned DEFAULT NULL,
  `fare` float unsigned DEFAULT NULL,
  `sex` enum('M','F') NOT NULL,
  `sibsp` tinyint unsigned DEFAULT NULL,
  `pclass` enum('1','2','3') NOT NULL,
  `embarked` tinyint DEFAULT NULL,
  `survived` enum('Y','N') DEFAULT NULL,
  PRIMARY KEY (`passengerid`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci;
```

In [10]:
import pandas as pd

# Cargar el CSV en un DataFrame (separador por punto y coma)
archivo = 'Titanic-Sencillo.csv'
df = pd.read_csv(archivo, sep=';')

# Mostrar información del DataFrame
print(df.info())
print("\nPrimeras filas:")
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Passengerid  1309 non-null   int64  
 1   Age          1309 non-null   int64  
 2   Fare         1309 non-null   int64  
 3   Sex          1309 non-null   int64  
 4   sibsp        1309 non-null   int64  
 5   Pclass       1309 non-null   int64  
 6   Embarked     1307 non-null   float64
 7   Survived     1309 non-null   int64  
dtypes: float64(1), int64(7)
memory usage: 81.9 KB
None

Primeras filas:
   Passengerid  Age    Fare  Sex  sibsp  Pclass  Embarked  Survived
0            1   22     725    0      1       3       2.0         0
1            2   38  712833    1      1       1       0.0         1
2            3   26    7925    1      0       3       2.0         1
3            4   35     531    1      1       1       2.0         1
4            5   35     805    0      0       3       2.0         0


In [11]:
import mysql.connector
from mysql.connector import Error

# Configurar conexión a MySQL
config = {
    'host': 'localhost',      # Cambiar si es necesario
    'port': 3307,             # Puerto personalizado
    'user': 'root',           # Cambiar con tu usuario
    'password': 'SZfetCPQA2C8ROSOvUp1',   # Cambiar con tu contraseña
    'database': 'titanic'
}

# Preparar datos para insertar
# Convertir valores y mapear columnas
df_insert = df.copy()

# Convertir valores de supervivencia a Y/N (convertir a int primero desde float)
if 'Survived' in df_insert.columns:
    df_insert['Survived'] = df_insert['Survived'].astype(int).map({1: 'Y', 0: 'N'})

# Mapear sexo: 0 = M (Male), 1 = F (Female) (convertir a int primero desde float)
if 'Sex' in df_insert.columns:
    df_insert['Sex'] = df_insert['Sex'].astype(int).map({0: 'M', 1: 'F'})

print("Datos listos para insertar:")
print(df_insert.head())

Datos listos para insertar:
   Passengerid  Age    Fare Sex  sibsp  Pclass  Embarked Survived
0            1   22     725   M      1       3       2.0        N
1            2   38  712833   F      1       1       0.0        Y
2            3   26    7925   F      0       3       2.0        Y
3            4   35     531   F      1       1       2.0        Y
4            5   35     805   M      0       3       2.0        N


In [12]:
try:
    # Conectar a la base de datos
    conexion = mysql.connector.connect(**config)
    cursor = conexion.cursor()
    
    # SQL INSERT
    sql_insert = """INSERT INTO pasajeros 
                    (passengerid, age, fare, sex, sibsp, pclass, embarked, survived) 
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)"""
    
    print("SQL INSERT:")
    print(sql_insert)
    print("\n" + "="*80 + "\n")
    
    # Insertar fila por fila
    insertados = 0
    errores = 0
    errores_lista = []
    
    for index, row in df_insert.iterrows():
        try:
            # Obtener valores con los nombres correctos de columnas
            passengerid = int(row['Passengerid'])
            print(f"Fila {index}: passengerid={passengerid}", end="")
            
            age = int(row['Age']) if pd.notna(row['Age']) else None
            print(f", age={age}", end="")
            
            fare = float(row['Fare']) if pd.notna(row['Fare']) else None
            print(f", fare={fare}", end="")
            
            sex = row['Sex']  # Ya está mapeado a M/F
            print(f", sex={sex}", end="")
            
            sibsp = int(row['sibsp']) if pd.notna(row['sibsp']) else None
            print(f", sibsp={sibsp}", end="")
            
            pclass = str(int(row['Pclass']))  # Convertir a int primero luego a string
            print(f", pclass={pclass}", end="")
            
            embarked = int(row['Embarked']) if pd.notna(row['Embarked']) else None
            print(f", embarked={embarked}", end="")
            
            survived = row['Survived']  # Ya está mapeado a Y/N
            print(f", survived={survived}")
            
            valores = (passengerid, age, fare, sex, sibsp, pclass, embarked, survived)
            
            cursor.execute(sql_insert, valores)
            insertados += 1
            print(f"  ✓ Insertado correctamente\n")
            
        except Exception as e:
            errores += 1
            error_msg = f"Fila {index}: {str(e)}"
            errores_lista.append(error_msg)
            print(f"\n  ✗ Error en fila {index}: {e}\n")
            # Continuar sin detener
    
    conexion.commit()
    print("="*80)
    print(f"\n✓ {insertados} registros insertados correctamente.")
    print(f"✗ {errores} errores durante la inserción.")
    
    if errores_lista:
        print("\nDetalle de errores:")
        for error in errores_lista:
            print(f"  - {error}")
    
    cursor.close()
    conexion.close()
except Error as err:
    print(f"Error de conexión: {err}")

SQL INSERT:
INSERT INTO pasajeros 
                    (passengerid, age, fare, sex, sibsp, pclass, embarked, survived) 
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)


Fila 0: passengerid=1, age=22, fare=725.0, sex=M, sibsp=1, pclass=3, embarked=2, survived=N

  ✗ Error en fila 0: 1062 (23000): Duplicate entry '1' for key 'pasajeros.PRIMARY'

Fila 1: passengerid=2, age=38, fare=712833.0, sex=F, sibsp=1, pclass=1, embarked=0, survived=Y

  ✗ Error en fila 1: 1062 (23000): Duplicate entry '2' for key 'pasajeros.PRIMARY'

Fila 2: passengerid=3, age=26, fare=7925.0, sex=F, sibsp=0, pclass=3, embarked=2, survived=Y

  ✗ Error en fila 2: 1062 (23000): Duplicate entry '3' for key 'pasajeros.PRIMARY'

Fila 3: passengerid=4, age=35, fare=531.0, sex=F, sibsp=1, pclass=1, embarked=2, survived=Y

  ✗ Error en fila 3: 1062 (23000): Duplicate entry '4' for key 'pasajeros.PRIMARY'

Fila 4: passengerid=5, age=35, fare=805.0, sex=M, sibsp=0, pclass=3, embarked=2, survived=N

  ✗ Error en


  ✗ Error en fila 42: 1062 (23000): Duplicate entry '43' for key 'pasajeros.PRIMARY'

Fila 43: passengerid=44, age=3, fare=415792.0, sex=F, sibsp=1, pclass=2, embarked=0, survived=Y

  ✗ Error en fila 43: 1062 (23000): Duplicate entry '44' for key 'pasajeros.PRIMARY'

Fila 44: passengerid=45, age=19, fare=78792.0, sex=F, sibsp=0, pclass=3, embarked=1, survived=Y

  ✗ Error en fila 44: 1062 (23000): Duplicate entry '45' for key 'pasajeros.PRIMARY'

Fila 45: passengerid=46, age=28, fare=805.0, sex=M, sibsp=0, pclass=3, embarked=2, survived=N

  ✗ Error en fila 45: 1062 (23000): Duplicate entry '46' for key 'pasajeros.PRIMARY'

Fila 46: passengerid=47, age=28, fare=155.0, sex=M, sibsp=1, pclass=3, embarked=1, survived=N

  ✗ Error en fila 46: 1062 (23000): Duplicate entry '47' for key 'pasajeros.PRIMARY'

Fila 47: passengerid=48, age=28, fare=775.0, sex=F, sibsp=0, pclass=3, embarked=1, survived=Y

  ✗ Error en fila 47: 1062 (23000): Duplicate entry '48' for key 'pasajeros.PRIMARY'

Fila